# Data Transform

In [149]:
# import libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [150]:
# define paths to parquet files

names_countries = "../data/silver/df_join_fn_cc_cleaned/"


In [151]:
# set a spark session

def create_spark_session(app_name:str):
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .getOrCreate()
    return spark

In [152]:
spark = create_spark_session("gold_pipeline")

In [153]:
df_names_country = spark.read.parquet(names_countries)
df_names_country.show(5)

+--------+------+-------+-----+------------+------------+
|forename|gender|country|count|country_code|country_name|
+--------+------+-------+-----+------------+------------+
|   Zeret|     M|     PS|    9|          PS|   palestine|
|   Zeret|     M|     PS|    9|          PS|   palestine|
|    عآزف|     M|     PS|    9|          PS|   palestine|
|    عآزف|     M|     PS|    9|          PS|   palestine|
|  Mhmood|     F|     PS|    9|          PS|   palestine|
+--------+------+-------+-----+------------+------------+
only showing top 5 rows


## 1. Adapting the cleaned data for Deep Learning model

In [154]:
# Define a list with LATAM country names, to seek it on country_naem dataframe
"""
In this spet I ensure that the resulting name match with LATAM names
"""
latam_countries = [
    "argentina", "bolivia", "brazil", "chile", "colombia", "costa rica",
    "cuba", "ecuador", "el salvador", "guatemala", "honduras", "mexico",
    "nicaragua", "panama", "paraguay", "peru", "puerto rico", "dominican republic",
    "uruguay", "venezuela"
]

df_latam = df_names_country.filter(col('country_name').rlike("|".join(latam_countries)))
df_latam.show()

+--------+------+-------+-----+------------+------------+
|forename|gender|country|count|country_code|country_name|
+--------+------+-------+-----+------------+------------+
|    Jose|     M|     PA|23250|          PA|      panama|
|    Jose|     M|     PA|23250|          PA|      panama|
|    Luis|     M|     PA|20265|          PA|      panama|
|    Luis|     M|     PA|20265|          PA|      panama|
|  Carlos|     M|     PA|17063|          PA|      panama|
|  Carlos|     M|     PA|17063|          PA|      panama|
|   Maria|     F|     PA|12249|          PA|      panama|
|   Maria|     F|     PA|12249|          PA|      panama|
|    Juan|     M|     PA|11510|          PA|      panama|
|    Juan|     M|     PA|11510|          PA|      panama|
|   Jorge|     M|     PA|10143|          PA|      panama|
|   Jorge|     M|     PA|10143|          PA|      panama|
|     Ana|     F|     PA| 7962|          PA|      panama|
|     Ana|     F|     PA| 7962|          PA|      panama|
|  Manuel|    

In [155]:
# Count the number of records on latam name dataframe
"""
As a result we got 1,447,779 records with LATAM names
"""
df_latam.count()

7238895

In [156]:
# Resume how many records has every country code
df_latam.groupBy('country_name').count().sort(col("count").desc()).show()

+------------+-------+
|country_name|  count|
+------------+-------+
|    colombia|1659585|
|      mexico|1154000|
|        peru|1119650|
|      brazil| 942110|
|       chile| 757655|
|     bolivia| 375525|
|   argentina| 279755|
|      panama| 274375|
|   guatemala| 229885|
|  costa rica| 190205|
|     uruguay| 158060|
|     ecuador|  57795|
| puerto rico|  31865|
|    honduras|   6200|
| el salvador|   2230|
+------------+-------+



In [157]:
df_gender = df_latam.groupBy('country_name', 'gender').count().sort(col('gender').desc(), col('count').desc())
df_gender.show()

+------------+------+------+
|country_name|gender| count|
+------------+------+------+
|    colombia|     M|831180|
|        peru|     M|609530|
|      mexico|     M|577675|
|      brazil|     M|476920|
|       chile|     M|378315|
|     bolivia|     M|207535|
|   argentina|     M|140670|
|   guatemala|     M|128950|
|      panama|     M|128735|
|  costa rica|     M| 96055|
|     uruguay|     M| 77250|
|     ecuador|     M| 30410|
| puerto rico|     M| 13300|
|    honduras|     M|  3045|
| el salvador|     M|  1040|
|    colombia|     F|828405|
|      mexico|     F|576325|
|        peru|     F|510120|
|      brazil|     F|465190|
|       chile|     F|379340|
+------------+------+------+
only showing top 20 rows


In [158]:
df_gender.count()

30

### a. Remove unnecessary columns and change columns name for neural network training

In [159]:
# Removils unnecessary columns

df_latam_nn = df_latam.drop("country", "count", "country_code", "country_name")
df_latam_nn.show()

+--------+------+
|forename|gender|
+--------+------+
|    Jose|     M|
|    Jose|     M|
|    Luis|     M|
|    Luis|     M|
|  Carlos|     M|
|  Carlos|     M|
|   Maria|     F|
|   Maria|     F|
|    Juan|     M|
|    Juan|     M|
|   Jorge|     M|
|   Jorge|     M|
|     Ana|     F|
|     Ana|     F|
|  Manuel|     M|
|  Manuel|     M|
|  Javier|     M|
|  Javier|     M|
| Ricardo|     M|
| Ricardo|     M|
+--------+------+
only showing top 20 rows


In [160]:
df_latam_nn = df_latam_nn.withColumnRenamed("forename", "NAME")
df_latam_nn = df_latam_nn.withColumnRenamed("gender", "GENDER")
df_latam_nn.show()


+-------+------+
|   NAME|GENDER|
+-------+------+
|   Jose|     M|
|   Jose|     M|
|   Luis|     M|
|   Luis|     M|
| Carlos|     M|
| Carlos|     M|
|  Maria|     F|
|  Maria|     F|
|   Juan|     M|
|   Juan|     M|
|  Jorge|     M|
|  Jorge|     M|
|    Ana|     F|
|    Ana|     F|
| Manuel|     M|
| Manuel|     M|
| Javier|     M|
| Javier|     M|
|Ricardo|     M|
|Ricardo|     M|
+-------+------+
only showing top 20 rows


In [161]:
def count_null_values(df):
    for c in df.columns:
        print(c, ": ", df.filter(col(c).isNull()).count())

In [162]:
count_null_values(df_latam_nn)

NAME :  0
GENDER :  0


In [163]:
# validate duplicates number
df_latam_nn.count() - df_latam_nn.dropDuplicates().count()

6407695

In [164]:
# remove duplicates from dataframe with latam names
df_latam_nn = df_latam_nn.dropDuplicates()

In [165]:
print(df_latam_nn.filter(df_latam_nn["NAME"].isNull()).count())

0


In [166]:
df_latam_nn.count()

831200

In [167]:
df_latam_nn.printSchema()

root
 |-- NAME: string (nullable = true)
 |-- GENDER: string (nullable = true)



### b. Save dataframe adapted to neural network

In [ ]:
df_latam_nn.write.mode("overwrite").parquet("../data/gold/df_latam_names_for_nn_training")